<div class="panel panel-success">
    <div class="panel-heading">
        <b> Eklärung - Das Notebook Starten</b>
    </div>
    <div class="panel-body">

**Um das Notebook zu starten**: 
        
1. Klicke oben links in der Menüleiste auf **"Run"** und anschließend auf **"Run All Cells"**
        
**Falls ein Neustart nötig ist**:
        
1. Klicke oben links in der Menüleiste auf **"Kernel"** und anschließend auf **"Restart and Clear Outputs of All Cells"**

2. Klicke oben links in der Menüleiste auf **"Run"** und anschließend auf **"Run All Cells"**

# Daten importieren

In [ ]:
from treech import ClassificationTree as ct

import pandas as pd
import numpy as np

from ipywidgets import interact, interactive, fixed, interact_manual, Layout, HBox
import ipywidgets as widgets

import plotly as py
from plotly.offline import iplot
import cufflinks as cf
cf.go_offline()
from IPython.display import display, HTML

<div class="panel panel-success">
    <div class="panel-heading">
        <b> Eklärung - Daten</b>
    </div>
    <div class="panel-body">

Hier siehst du zwei Datensätze, die weiter unten zum trainieren und bewerten eines Entscheidungsbaums genutzt werden.

* **Trainingsdatensatz**: Wird genutzt, um einen Entscheidungsbaum automatisch mit einem Algorithmus zu erstellen bzw. zu trainieren.
        
    
* **Testdatensatz**: Wird genutzt, um zu bewerten wie gut ein fertiger Entscheidungsbaum funktioniert.

In [ ]:
def read_csv():
    df_train = pd.read_csv('Daten/Training/jim_binaer.csv', sep = ';')
    df_test = pd.read_csv('Daten/Test/jim_binaer.csv', sep = ';')
    print('Trainingsdaten')
    display(df_train)
    print('Testdaten')
    display(df_test)
    return df_train, df_test

data_widget=interactive(read_csv)
#data_widget.result = pd.DataFrame(columns=['Importiere Datensatz'])
data_widget

# Entscheidungsbaum trainieren

<div class="panel panel-success">
    <div class="panel-heading">
        <b> Eklärung - Einen Entscheidungsbaum trainieren</b>
    </div>
    <div class="panel-body">

Mit dem folgenden Menü kannst du einen Entscheidungsbaum trainieren. D. h. ein Algorithmus wählt basierend auf den Trainingsdaten nach und nach Variablen aus und erstellt mit diesen automatisch einen Entscheidungsbaum. Du hast hier als Mensch aber eine Möglichkeit einzugreifen, um das Ergebnis zu beeinflussen. Du kannst die Zielvariable wählen und außerdem einen Parameter bestimmen. Dieser wird beim Maschinellen Lernen auch Hyperparamter genannt.
        
Der hier aufgeführte Hyperparameter bestimmt wann der Trainingsprozess abbricht:
        
* **Max_Tiefe**: Dieser Parameter bestimmt was die maximale Tiefe des Entscheidungsbaums sein darf. Tiefe bedeutet die Anzahl der Regel-Knoten in einem Pfad des Baums. Wenn Max_Tiefe auf 3 gesetzt wird bedeutet das also, dass in jedem Pfad des Baums maximal 3 Entscheidungsregeln erlaubt sind. Sind drei Entscheidungsregeln in einem Pfad erreicht, bricht der Prozess in diesem Pfad ab. Welche Variablen für die Entscheidungsregeln genutzt werden, kann man allerdings mit diesem Parameter nicht beeinflussen. 
        

In [ ]:
def grow_tree(Zielvariable,  Max_Tiefe):
    
    Krit = 'misclassification_error'
    
    tree = ct.DecisionTree(data = data_widget.result[0], target = Zielvariable, crit = Krit)
    tree.grow_tree(max_depth = Max_Tiefe, crit = Krit)
    tree.print_tree()
    display(tree.tree_graph)

    display(HTML("<b>Rate der korrekten Klassifikationen:</b>"))
    print('Trainingsdaten',round((tree.prediction_accuracy(data_widget.result[0]))*100,1),'% \n')
    print('Testdaten',round(tree.prediction_accuracy(data_widget.result[1])*100,1),'% \n') 
    
    display(HTML("<b>Weitere Merkmale des Entscheidungsbaums:</b>"))

    a=tree.get_status()
    
    tree_info = pd.DataFrame({
        'Merkmal': [
            'Tatsächliche Tiefe',
            'Anzahl Knoten',
            'Anzahl Endknoten',
            'Durchschnittliche Anzahl Fälle pro Endknoten'
        ],
        'Wert': [
            int(a['max_depth']) - 1,
            int(a['node_count']),
            int(a['leaf_node_count']),
            a['mean_leaf_cases']
        ]
    })

    display(tree_info.style.hide(axis="index").format({'Wert': lambda x: f'{int(x)}' if x == int(x) else f'{x:.1f}'}))

    return tree
    
tree_widget = interactive(grow_tree,{'manual': True, 'manual_name': 'Erstelle Baum'}, Zielvariable = data_widget.result[0].columns.sort_values(), Max_Tiefe=(0,20,1), ) 
tree_widget


## Verlauf des Trainingsprozesses Visualisieren

In [ ]:
def analyse_training():
    tree_widget.result.evaluation_depth(data_widget.result[0],data_widget.result[1]).iplot(xTitle='Tiefe des Baums', yTitle='Rate korrekter Klassifikationen')
   
analyse_tree_widget = interactive(analyse_training,{'manual': True, 'manual_name': 'Verlauf analysieren'})
analyse_tree_widget


## Entscheidungsbaum Ausgeben

In [ ]:
def output_tree():       
    tree_widget.result.tree_graph.view()
    
output_tree_widget = interactive(output_tree,{'manual': True, 'manual_name': 'Tree ausgeben'})
output_tree_widget
